# Classificador de Intenções Local e Offline para a Assistente MINA

Este notebook demonstra o funcionamento do sistema de classificação de intenções e consulta ao banco de dados SQLite acadêmico desenvolvido para a assistente virtual **MINA** (UNESP Sorocaba).

### Como funciona:
1. O usuário digita ou fala uma pergunta.
2. A pergunta é interceptada localmente por expressões regulares otimizadas.
3. Se corresponder a um padrão acadêmico conhecido, o sistema faz a query direta no banco de dados SQLite local (`academic.db`).
4. A resposta é retornada instantaneamente e de forma 100% offline, pulando a chamada ao LLM.

## 1. Configurando o Banco de Dados Acadêmico (academic.db)
Vamos criar o arquivo SQLite local e inserir alguns dados fictícios idênticos aos coletados pelos scrapers do projeto MINA na UNESP.

In [ ]:
import sqlite3
import os
from datetime import datetime

DB_PATH = "academic.db"

def inicializar_banco():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Criar tabelas
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS professors (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE,
            room TEXT NOT NULL,
            email TEXT,
            department TEXT
        )
    """)
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS schedules (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            subject TEXT NOT NULL,
            weekday INTEGER NOT NULL,
            start_time TEXT NOT NULL,
            end_time TEXT NOT NULL,
            room TEXT NOT NULL,
            teacher_name TEXT
        )
    """)
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS news_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            is_event BOOLEAN DEFAULT 0
        )
    """)
    
    # Inserir dados de teste
    cursor.execute("INSERT OR REPLACE INTO professors (name, room, department) VALUES ('Prof. Eduardo', 'Sala 01 - Depto. ECA', 'Controle e Automação')")
    cursor.execute("INSERT OR REPLACE INTO professors (name, room, department) VALUES ('Prof. Alexandre Simões', 'Sala 04 - Depto. ECA', 'Robótica')")
    
    cursor.execute("INSERT OR REPLACE INTO schedules (subject, weekday, start_time, end_time, room, teacher_name) VALUES ('Cálculo Diferencial I', 0, '08:00', '10:00', 'Sala 1 (Bloco A)', 'Profa. Maria')")
    cursor.execute("INSERT OR REPLACE INTO schedules (subject, weekday, start_time, end_time, room, teacher_name) VALUES ('Eletrônica Industrial', 0, '10:00', '12:00', 'Laboratório 2', 'Prof. Roberto')")
    
    cursor.execute("INSERT OR REPLACE INTO news_events (title, is_event) VALUES ('Palestra sobre Inteligência Artificial no Anfiteatro', 1)")
    cursor.execute("INSERT OR REPLACE INTO news_events (title, is_event) VALUES ('Inscrições abertas para o Hackathon UNESP', 0)")

    conn.commit()
    conn.close()
    print("[+] Banco de dados academic.db criado e populado com sucesso!")

inicializar_banco()

## 2. Definindo a Classe do Classificador de Intenções (IntentClassifier)
Esta classe contém a lógica de expressões regulares para interceptar as perguntas e rodar as consultas SQLite correspondentes.

In [ ]:
import re
from typing import Tuple, Optional

class IntentClassifier:
    def __init__(self):
        # Regras de expressões regulares para detectar as intenções
        self.rules = {
            "sala_professor": re.compile(
                r"\b(sala|gabinete|onde fica|onde atende|onde encontrar)\b.*\b(professor|professora|prof|profa)\b",
                re.IGNORECASE
            ),
            "horario_aulas": re.compile(
                r"\b(aula|horario|horário|cronograma|agenda|matéria|materia|grade)\b",
                re.IGNORECASE
            ),
            "noticias": re.compile(
                r"\b(notícia|noticia|notícias|noticias|evento|eventos|mural|novidade|novidades|jornal)\b",
                re.IGNORECASE
            ),
            "cantina": re.compile(
                r"\b(cantina|salgado|chipa|intervalo|comer|alimentação|refeição|fome)\b",
                re.IGNORECASE
            )
        }

    def classify_and_execute(self, user_query: str) -> Tuple[bool, Optional[str]]:
        query_clean = user_query.strip().lower()
        if not query_clean:
            return False, None

        if self.rules["sala_professor"].search(query_clean):
            return True, self._handle_sala_professor(query_clean)

        if self.rules["horario_aulas"].search(query_clean):
            return True, self._handle_horario_aulas(query_clean)

        if self.rules["noticias"].search(query_clean):
            return True, self._handle_noticias(query_clean)

        if self.rules["cantina"].search(query_clean):
            return True, self._handle_cantina()

        return False, None

    def _handle_sala_professor(self, query: str) -> str:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT name, room, department FROM professors")
        professores = [{"name": r[0], "room": r[1], "department": r[2]} for r in cursor.fetchall()]
        conn.close()

        nome_procurado = ""
        palavras = query.split()
        for i, pal in enumerate(palavras):
            if pal in ["prof", "professor", "professora", "profa", "do", "da"] and i + 1 < len(palavras):
                nome_procurado = palavras[i + 1]
                break

        if nome_procurado:
            for p in professores:
                if nome_procurado in p["name"].lower():
                    return f"O {p['name']} atende na {p['room']} do departamento de {p['department']}."

        lista_profs = ", ".join([f"{p['name']} ({p['room']})" for p in professores])
        return f"Não identifiquei o nome do professor com clareza. Professores cadastrados: {lista_profs}."

    def _handle_horario_aulas(self, query: str) -> str:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        # Simula as aulas de hoje (Segunda-feira = 0)
        cursor.execute("SELECT subject, start_time, end_time, room, teacher_name FROM schedules WHERE weekday = 0")
        aulas = [{"subject": r[0], "start_time": r[1], "end_time": r[2], "room": r[3], "teacher_name": r[4]} for r in cursor.fetchall()]
        conn.close()

        if not aulas:
            return "Não há aulas agendadas para hoje."

        resposta = "Aulas de hoje: "
        lista = [f"{a['subject']} na {a['room']} das {a['start_time']} às {a['end_time']} com {a['teacher_name']}" for a in aulas]
        return resposta + "; ".join(lista) + "."

    def _handle_noticias(self, query: str) -> str:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT title, is_event FROM news_events ORDER BY id DESC LIMIT 3")
        noticias = [{"title": r[0], "is_event": r[1]} for r in cursor.fetchall()]
        conn.close()

        if not noticias:
            return "Nenhum evento ou notícia cadastrado no momento."

        lista = []
        for i, n in enumerate(noticias):
            tipo = "evento" if n["is_event"] else "notícia"
            lista.append(f"{i+1}: [{tipo}] {n['title']}")
        return "Novidades no mural da UNESP: " + " | ".join(lista)

    def _handle_cantina(self) -> str:
        return "A chipa quentinha e os salgados saem na cantina principal da UNESP Sorocaba nos intervalos das 9 horas da manhã e das 20h30 da noite."

## 3. Teste Interativo Local (Simulação de CLI)
Rode a célula abaixo e digite suas perguntas para simular o comportamento local da assistente MINA. Experimente:
* *"qual a sala do professor eduardo?"*
* *"quais as notícias do mural?"*
* *"que horas sai o salgado na cantina?"*
* *"qualquer outra pergunta fora de escopo"*

In [ ]:
classifier = IntentClassifier()

def testar_mina(pergunta):
    print(f"Usuário: {pergunta}")
    
    # Executa a classificação local
    detectado, resposta_local = classifier.classify_and_execute(pergunta)
    
    if detectado:
        print(f"Mina (Local / Offline): {resposta_local}")
    else:
        print("Mina (Fallback -> Enviar para o LLM externo na nuvem...)")
    print("-" * 70)

# Rodando bateria de testes rápidos
testar_mina("onde fica o gabinete do professor eduardo?")
testar_mina("qual o horário das aulas de hoje?")
testar_mina("me dê notícias sobre o mural da unesp")
testar_mina("tô com fome, o que tem de bom na cantina?")
testar_mina("qual a distância da Terra até a Lua?") # Deve cair no LLM